# Databricks Notebook: Banking Medallion ETL Workflow
This notebook demonstrates source to Bronze, Silver, and Gold transformations.

In [1]:
import os, json
from pyspark.sql import functions as F

# Config params
CONFIG_FILE = 's3://banking-datalake/configs/etl_settings.json'
FILE_PATH = 'hdfs:///data/banking/landing/daily_feed.csv'

dbutils.widgets.text('env', 'prod')
env = dbutils.widgets.get('env')

In [2]:
# UDF Definition
@udf(returnType='string')
def calculate_credit_tier(score):
    if score is None: return 'UNKNOWN'
    return 'PRIME' if score > 750 else 'SUBPRIME'

spark.udf.register('udf_credit_tier', calculate_credit_tier)

In [3]:
# Bronze to Silver Ingestion
df_raw = spark.read.option('header', 'true').csv(FILE_PATH)

df_silver = df_raw.withColumn('tier', calculate_credit_tier(F.col('credit_score')))
df_silver.write.mode('overwrite').saveAsTable('banking_catalog.silver.cleansed_transactions')

# SQL Execution with CTE and Joins
spark.sql("""
    WITH temp_gold_agg AS (
        SELECT customer_id, SUM(amount) as total_amt
        FROM banking_catalog.silver.cleansed_transactions
        GROUP BY customer_id
    )
    CREATE TABLE IF NOT EXISTS banking_catalog.gold.executive_kpi_dashboard AS
    SELECT a.customer_id, a.total_amt, c.tier
    FROM temp_gold_agg a
    INNER JOIN banking_catalog.silver.verified_customer_accounts c ON a.customer_id = c.customer_id
    UNION
    SELECT customer_id, total_amt, 'VIP' as tier
    FROM banking_catalog.silver.vip_deposits;
""")